# Exercise 2: Landsat Satellite Image Classification
## Remote Sensing & Crater Detection — BigQuery ML
**Өгөгдөл:** 9,608,135 Landsat зураг (1972-2021)
**Зорилго:** Үүлэрхэг болон цэлмэг зургийг ангилах ML загвар бүтээх
> Энэ notebook нь Google BigQuery-с шууд өгөгдөл дуудаж,
> локал компьютерт татахгүйгээр үүлэн орчинд ML хийнэ.

In [ ]:
# Шаардлагатай library-уудыг суулгах
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')
print('✅ Library-ууд амжилттай ачааллагдлаа!')

## 1. BigQuery-с Landsat өгөгдөл дуудах
9.6 сая зургаас статистик мэдээллийг үүлэн орчноос шууд дуудна.

In [ ]:
# Манай системийн Backend API-аас BigQuery өгөгдөл дуудах
API_URL = 'https://space-learning-system.onrender.com'

# Landsat статистик
resp = requests.get(f'{API_URL}/api/bigquery/landsat-stats')
stats = resp.json()['data']
print('📡 Landsat өгөгдлийн статистик:')
print(f'  Нийт зургийн тоо: {stats["total_scenes"]:,}')
print(f'  Дагуулын тоо: {stats["satellites"]}')
print(f'  Хамгийн эрт: {stats["earliest"]["value"]}')
print(f'  Хамгийн сүүлийн: {stats["latest"]["value"]}')
print(f'  Дундаж үүлэрхэг байдал: {stats["avg_cloud_cover"]}%')

In [ ]:
# Дагуулаар ангилсан өгөгдөл
resp2 = requests.get(f'{API_URL}/api/bigquery/landsat-by-satellite')
sat_data = resp2.json()['data']
df = pd.DataFrame(sat_data)
print('\n🛰️ Дагуул бүрийн өгөгдөл:')
print(df.to_string(index=False))

## 2. Өгөгдөл бэлдэх — Feature Engineering

In [ ]:
# ML-д зориулж feature үүсгэх
df['cloud_category'] = pd.cut(
    df['avg_cloud_cover'].astype(float),
    bins=[0, 20, 50, 100],
    labels=['Цэлмэг', 'Хэсэгчлэн үүлэрхэг', 'Үүлэрхэг']
)

df['scene_count'] = df['scene_count'].astype(int)
df['avg_cloud_cover'] = df['avg_cloud_cover'].astype(float)

# Synthetic features нэмэх
np.random.seed(42)
n = len(df)
df['solar_elevation'] = np.random.uniform(20, 70, n)
df['sensor_quality'] = np.random.uniform(0.7, 1.0, n)

print('✅ Feature-ууд бэлэн:')
print(df[['spacecraft_id', 'scene_count', 'avg_cloud_cover', 'cloud_category']].to_string())

## 3. Random Forest ML Загвар

In [ ]:
# Landsat ML үр дүнгийн өгөгдөл дуудах
resp3 = requests.get(f'{API_URL}/api/bigquery/landsat-ml')
ml_data = resp3.json()['data']
df_ml = pd.DataFrame(ml_data)

# Feature болон label
X = df_ml[['cloud_cover', 'solar_elevation_angle']].fillna(0)
y = (df_ml['cloud_cover'].astype(float) < 30).astype(int)  # 1=цэлмэг, 0=үүлэрхэг

# Train/test хуваах
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Random Forest загвар
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Үнэлэх
y_pred = rf.predict(X_test)
print('🤖 Random Forest ML Үр дүн:')
print(classification_report(y_test, y_pred,
      target_names=['Үүлэрхэг', 'Цэлмэг']))

## 4. Үр дүнг дүрслэх

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Дагуул бүрийн зургийн тоо
axes[0].bar(df['spacecraft_id'], df['scene_count'], color='steelblue')
axes[0].set_title('Дагуул бүрийн зургийн тоо\n(BigQuery: 9.6M нийт)')
axes[0].set_xlabel('Дагуул')
axes[0].set_ylabel('Зургийн тоо')
axes[0].tick_params(axis='x', rotation=45)

# Feature importance
importances = rf.feature_importances_
axes[1].bar(X.columns, importances, color='orange')
axes[1].set_title('Feature Importance\n(Random Forest)')
axes[1].set_xlabel('Feature')
axes[1].set_ylabel('Importance')

plt.tight_layout()
plt.savefig('landsat_ml_results.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Дүрслэл хадгалагдлаа!')

## 5. Дүгнэлт
- **9,608,135** Landsat зургийн мета өгөгдлийг BigQuery-с үүлэн орчноос шууд дуудлаа
- Локал компьютерт **ямар ч өгөгдөл татаагүй**
- Random Forest загвар үүлэн орчинд амжилттай ажиллалаа
- Энэ нь **Pangeo загварын** 'өгөгдөл үүлэнд, тооцоолол үүлэнд' зарчмыг хэрэгжүүлсэн